# AI Textbook Tutor: all-in-one Google Colab notebook
The whole project is bundled inside this notebook. Run the cells from top to bottom; the only thing you upload is your books (text-based PDF or EPUB).
Colab sessions are temporary: save your work to Google Drive (cell 9) before you leave.

In [ ]:
#@title 0. Bundled project files (no need to open this)
PAYLOAD = """
...
"""
print(f"Bundled project: {len(PAYLOAD) // 1000} KB of data ready")

In [ ]:
#@title 1. Install AI Textbook Tutor (everything is bundled in this notebook)
import base64, importlib, io, os, zipfile
ROOT = os.getcwd()
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(ROOT)
os.chdir(f"{ROOT}/ai-textbook-tutor")
!pip install -q -r requirements.txt
!pip install -q --force-reinstall --no-deps .
importlib.invalidate_caches()
from google.colab import files
!aitutor --help | head -3

In [ ]:
#@title 2. Upload your books. Name them like biology_g8_en.pdf (subject_g<grade>_<language>)
import shutil
os.makedirs("books", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"books/{name}")
!ls -l books

In [ ]:
#@title 3. Import the books
!aitutor import books --out data/passages.jsonl
!aitutor books

In [ ]:
#@title 4. Ask the textbook
import subprocess
question = "How do plants make food?"  #@param {type:"string"}
subject = ""  #@param {type:"string"}
grade = 0  #@param {type:"integer"}
cmd = ["aitutor", "ask", question] + (["--subject", subject] if subject else []) + (["--grade", str(grade)] if grade else [])
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

In [ ]:
#@title 5. Adaptive quiz (type the number of your answer; leave empty to stop)
from pathlib import Path
from aitutor import ingest
from aitutor.quiz import QuizSession, generate
from aitutor.retriever import BM25Index
ps = ingest.load(Path("data/passages.jsonl"))
session = QuizSession(generate(ps, BM25Index(ps))[:5])
while (q := session.next()):
    print(f"\n[{q.topic}] {q.prompt}")
    for i, o in enumerate(q.options):
        print(f"  {i + 1}. {o}")
    try:
        choice = int(input("Your answer: ")) - 1
    except (ValueError, EOFError):
        break
    print("Correct." if session.answer(q, choice) else f"Not yet. Answer: {q.options[q.answer]} (see {q.source})")
print("Mastery estimates:", session.mastery())

## Optional: language model
Cells 6a to 6c add a small on-device model. It writes the answer from the retrieved passages, and AI Textbook Tutor removes every sentence the textbook does not support.
Start with English (small models are weak in Amharic, Afan Oromo and Somali), keep the answer language the same as the book's, and check the model's license.

In [ ]:
#@title 6a. Install the model engine (llama.cpp). Can take 5-10 minutes if it has to compile
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
!pip install -q huggingface_hub
import llama_cpp; print("llama-cpp-python", llama_cpp.__version__)

In [ ]:
#@title 6b. Download a small model (GGUF format). Change repo/file to use another model
from huggingface_hub import hf_hub_download, list_repo_files
MODEL_REPO = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"  #@param {type:"string"}
MODEL_FILE = "qwen2.5-1.5b-instruct-q4_k_m.gguf"  #@param {type:"string"}
try:
    MODEL_PATH = hf_hub_download(MODEL_REPO, MODEL_FILE, local_dir="models")
except Exception as e:
    print("Download failed:", e)
    try:
        print("GGUF files in that repo:", [f for f in list_repo_files(MODEL_REPO) if f.endswith(".gguf")])
    except Exception:
        print("Could not list the repo: check the repo name.")
    raise
print("Model saved to", MODEL_PATH)

In [ ]:
#@title 6c. Ask with the model (reuses the question from cell 4)
answer_language = "English"  #@param ["English", "Amharic", "Afan Oromo", "Somali"]
cmd = ["aitutor", "ask", question, "--model", MODEL_PATH, "--lang", answer_language] \
      + (["--subject", subject] if subject else []) + (["--grade", str(grade)] if grade else [])
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout or res.stderr[-1500:])

In [ ]:
#@title 7. Start the server inside Colab and use it from Python (teacher dashboard data and PDF report)
import subprocess, time, requests
PASSWORD = "choose-a-password"  #@param {type:"string"}
STUDENT_CODE = ""  #@param {type:"string"}  (access code for students; REQUIRED before you share a public link)
USE_MODEL = False  #@param {type:"boolean"}  (tick only after running 6a and 6b)
args = ["aitutor", "serve", "--admin-password", PASSWORD, "--port", "8000"]
if STUDENT_CODE:
    args += ["--student-code", STUDENT_CODE]
if USE_MODEL:
    args += ["--model", MODEL_PATH]
server = subprocess.Popen(args, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
BASE = "http://127.0.0.1:8000"
for _ in range(120 if USE_MODEL else 30):  # loading a model takes longer
    try:
        requests.get(BASE + "/health", timeout=1); break
    except requests.ConnectionError:
        time.sleep(1)
H = {"X-Student-Code": STUDENT_CODE} if STUDENT_CODE else {}
r = requests.post(BASE + "/api/ask", headers=H, json={"question": question, "subject": subject or None, "grade": grade or None}).json()
print("grounded:", r["grounded"], "| AI answer:", r["ai"], "| sentences removed:", r["removed"]); print(r["answer"]); print(r["hint"]); print([c["text"] for c in r["citations"]])
teacher = requests.Session()
assert teacher.post(BASE + "/api/login", json={"password": PASSWORD}).ok, "login failed"
print(teacher.get(BASE + "/analytics/summary").json())
open("ai_tutor_report.pdf", "wb").write(teacher.get(BASE + "/analytics/report.pdf").content)
files.download("ai_tutor_report.pdf")

In [ ]:
#@title 8. Get a temporary public web link (run cell 7 first; students open the first link, teachers the second)
assert PASSWORD != "choose-a-password", "Set a strong PASSWORD in cell 6 and run it again"
assert STUDENT_CODE, "Set a STUDENT_CODE in cell 7 and run it again: a public link without one shows your books to anyone"
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import re, subprocess, time
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url, deadline = None, time.time() + 60
while public_url is None and time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:   # cloudflared exited: stop waiting
        print("cloudflared stopped. Last message above.")
        break
    print(line.rstrip()[:200]) if "ERR" in line or "failed" in line else None
    m = re.search(r"https://(?!api\.)[-a-z0-9]+\.trycloudflare\.com", line)   # (api.* is cloudflare's own server)
    public_url = m.group(0) if m else None
if public_url:
    print("Students:", public_url, "  (access code:", STUDENT_CODE + ")")
    print("Teachers:", public_url + "/teacher", "  (password: the one you set in cell 6)")
    print("Add books in the browser: " + public_url + "/teacher/library")
    print("The link stops working when this Colab session ends. To stop sharing now: tunnel.terminate()")
else:
    print("Could not get a link. Run the cell again, or use the Colab-only options.")

In [ ]:
#@title 9. Save your work to Google Drive (Colab forgets everything when the session ends)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/ai-textbook-tutor
!cp -r data books /content/drive/MyDrive/ai-textbook-tutor/
!cp -f aitutor.db /content/drive/MyDrive/ai-textbook-tutor/ 2>/dev/null; echo saved
# Next session: unzip ai-textbook-tutor.zip, install (cell 1), then copy data/ and books/ back from Drive.

In [ ]:
#@title 10. Build an offline content pack and download it (for SD card / USB)
!aitutor pack build --out school.tutorpack --version 1
files.download("school.tutorpack")

In [ ]:
#@title 11. Stop the server when finished
server.terminate()

In [ ]:
#@title 12. (Optional) Run the automated tests
!pip install -q pytest httpx
!python -m pytest -q